## Day 1 — Train / Validation / Test Splits

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [6]:
df = pd.read_csv('Housing Prices/housing.csv')

In [7]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [8]:
from sklearn.preprocessing import LabelEncoder
lb = LabelEncoder()
df['ocean_proximity'] = lb.fit_transform(df['ocean_proximity'])

In [9]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,3
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,3
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,3
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,3
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,3


In [10]:
df.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [12]:
df["total_bedrooms"] = df["total_bedrooms"].fillna(df["total_bedrooms"].median())

In [13]:
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]


When training any model, you run into a problem: if you evaluate it on the 
same data it was trained on, it might look great while actually having just 
memorized the data rather than learned a real pattern. The usual fix is a 
train/test split — but if you also want to try different settings 
(hyperparameters), you need a third, neutral group to judge by, so you don't 
"contaminate" the final test data.

In [14]:
# Step 1: split off 60% train, 40% temp (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)

In [15]:
# Step 2: split the 40% temp into 20% val, 20% test (i.e. 50/50 of the temp set)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [16]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (12384, 9)
Validation: (4128, 9)
Test: (4128, 9)


In [17]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

Having picked the best max_depth using only the validation set, the real 
question now is: how will the model behave on data it has never seen at 
all — not during training, and not during hyperparameter selection either? 
This is exactly what the test set is for, and we touch it now for the first 
and only time.

In [18]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Try a few candidate values for max_depth
depth_candidates = [5, 10, 15, 20, None]  # None = unlimited depth
val_scores = []

for depth in depth_candidates:
    model = RandomForestRegressor(n_estimators=100, max_depth=depth, random_state=42)
    model.fit(X_train_scaled, y_train)
    
    val_preds = model.predict(X_val_scaled)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    val_scores.append(val_rmse)
    print(f"max_depth={depth}: Validation RMSE = {val_rmse:.2f}")

# Pick the best depth based on validation performance
best_depth = depth_candidates[np.argmin(val_scores)]
print(f"\nBest max_depth: {best_depth}")

max_depth=5: Validation RMSE = 68407.88
max_depth=10: Validation RMSE = 55076.78
max_depth=15: Validation RMSE = 52045.91
max_depth=20: Validation RMSE = 51784.58
max_depth=None: Validation RMSE = 51624.81

Best max_depth: None


### Hyperparameter Tuning: `max_depth` (Validation Set)

| max_depth | Validation RMSE |
|---|---|
| 5 | 68,407.88 |
| 10 | 55,076.78 |
| 15 | 52,045.91 |
| 20 | 51,784.58 |
| None | 51,624.81 |

**Best value: `max_depth=None`** (RMSE = 51,624.81). Error drops sharply up 
to depth 15, then plateaus — diminishing returns beyond that. Selected using 
only the validation set; test set remains untouched for final evaluation.

### task3: Evaluate the final model on the test set exactly once and report the score.

In [20]:
# Final model with the best hyperparameter found on the validation set
from sklearn.metrics import mean_absolute_error, r2_score
final_model = RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42)
final_model.fit(X_train_scaled, y_train)

# Evaluate on the test set — ONE TIME ONLY
test_preds = final_model.predict(X_test_scaled)

test_mae = mean_absolute_error(y_test, test_preds)
test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
test_r2 = r2_score(y_test, test_preds)

print(f"Test MAE: {test_mae:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")
print(f"Test R²: {test_r2:.4f}")

Test MAE: 33459.55
Test RMSE: 51339.26
Test R²: 0.8078


### Final Test Set Evaluation

**Model:** Random Forest Regressor (`max_depth=None`, `n_estimators=100`), 
selected using validation-set tuning.

| Metric | Score |
|---|---|
| MAE | 33,459.55 |
| RMSE | 51,339.26 |
| R² | 0.8078 |

The final model explains **~81% of the variance** in `median_house_value` 
on completely unseen test data, with predictions off by about $33,460 on 
average (MAE). This is consistent with the validation RMSE (~51,625), 
confirming the model generalizes well and the `max_depth=None` choice was 
sound — no signs of overfitting to the validation set.

This is the model's true, unbiased performance estimate, obtained by 
touching the test set exactly once after all tuning was complete.

If we had used the test set to choose max_depth instead of the validation 
set, we would have ended up selecting whichever setting happened to fit the 
test set best — not the setting that actually generalizes better. The 
reported score would have looked better than it truly is, because the model 
would have been indirectly shaped by data that was supposed to stay 
completely unseen — this is exactly what's known as data leakage.